In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d

# --- Third-party imports ---
from lifelines.utils import concordance_index

from sksurv.metrics import cumulative_dynamic_auc
from sksurv.util import Surv

from sklearn.metrics import brier_score_loss


In [2]:
sjmb03_pred = pd.read_csv('/hovestadtlab/sabina/methylation-survival/results/sjmb03_predictions.csv')
sjmb03_pred.set_index('Patient_ID', inplace=True) 
time_bins = [0.019167, 0.33333, 0.5847, 0.75, 0.94167, 1.1667, 1.5, 1.8333, 2.0833, 2.5833, 3.25, 3.9, 5.6667, 8.2875, 22.0]

surv_sj_high = pd.read_csv('/hovestadtlab/sabina/methylation-survival/data/surv_sj_high.csv')

In [7]:
surv_sj_high

,Unnamed: 0,Patient_ID,time,event,m,pfs_time_yrs,pfs_event
0,3,101279430012_R03C02,1.122519,1,1,0.684463,1
1,7,101279430012_R05C02,1.470226,1,1,1.218344,1
2,13,101279430024_R03C01,6.258727,0,0,6.258727,0
3,15,101279430024_R04C01,9.483915,0,1,9.483915,0
4,16,101279430024_R04C02,2.488706,1,1,2.488706,1
...,...,...,...,...,...,...,...
94,293,9934987115_R01C02,7.978097,0,1,7.978097,0
95,297,9934987115_R03C02,4.369610,1,1,4.369610,1
96,300,9934987115_R05C01,7.301848,0,1,7.301848,0
97,301,9934987115_R05C02,6.201232,0,1,6.201232,0


In [8]:
test_samples = sjmb03_pred.values #[high_risk,]
t_test_np = surv_sj_high['time'].values #[high_risk]
e_test_np = surv_sj_high['event'].values #[high_risk]

t_test_np = surv_sj_high['pfs_time_yrs'].values #[high_risk]
e_test_np = surv_sj_high['pfs_event'].values #[high_risk]

n_test_samples = test_samples.shape[0]
pred_times = np.concatenate(([0.0], time_bins))
eval_times = np.arange(0.1, 10.01, 0.1)


interpolated_probs = np.zeros((n_test_samples, len(eval_times)))
for i in range(n_test_samples):
    y_interp = test_samples[i, :]
    interp_func = interp1d(pred_times, y_interp, kind='linear', bounds_error=False, fill_value=(y_interp[0], y_interp[-1]))
    interpolated_probs[i, :] = interp_func(eval_times)

interpolated_probs = np.clip(interpolated_probs, 0.0, 1.0)

# --- 6b. Apply Smoothing
window = 10
kernel = np.ones(window) / window
pad_left = window // 2
pad_right = window - pad_left - 1
smoothed_interpolated_probs = np.zeros_like(interpolated_probs)

for i in range(n_test_samples):
    y = interpolated_probs[i, :]
    if np.isnan(y).any():
        smoothed_interpolated_probs[i, :] = np.nan
        continue
    y_padded = np.pad(y, (pad_left, pad_right), mode='reflect')
    y_conv = np.convolve(y_padded, kernel, mode='valid')
    y_mon = np.minimum.accumulate(y_conv)
    smoothed_interpolated_probs[i, :] = np.clip(y_mon, 0.0, 1.0)

# --- 7. Calculate Concordance Index (C-index) ---
print("\n--- Calculating C-index ---")
eval_time_cindex = 5.0; c_index = np.nan
idx_cindex = np.abs(eval_times - min(max(eval_times.min(), eval_time_cindex), eval_times.max())).argmin()

if idx_cindex < smoothed_interpolated_probs.shape[1]:
    probs_for_cindex = smoothed_interpolated_probs[:, idx_cindex]
    valid_mask_c1 = ~np.isnan(probs_for_cindex)
    if np.any(valid_mask_c1):
        risk_scores_cindex = -probs_for_cindex[valid_mask_c1] 
        t_test_c = t_test_np[valid_mask_c1]
        e_test_c = e_test_np[valid_mask_c1]
        try: c_index = concordance_index(t_test_c, -risk_scores_cindex, e_test_c.astype(bool))
        except Exception as e: print(f"Error C-index: {e}")
        print(f"C-index: {c_index:.4f}")
    else: print("Warning: All survival probs at t~5 NaN. Cannot calc C-index.")


--- Calculating C-index ---
C-index: 0.7118
